Using the scraped publications from raw_ees_pub_scrape add the contextual information that is helpful for elsewhere in the app.

At the moment this is hard coded based initially from a previous random excel project we did a while back, we'll need to make edits here as things change unless we can automate refreshed values (likely possible with release type if linking to EES admin tables).

I've added a test to compare row count to the raw scrape so it should ping if there are ever new publications that need their details adding.

In [0]:
%r
source("utils.R")

packages <- c("sparklyr", "DBI", "dplyr", "testthat", "arrow", "stringr")

install_if_needed(packages)
lapply(packages, library, character.only = TRUE)

scrape_table_name <- "catalog_40_copper_statistics_services.analytics_raw.ees_pub_scrape"
write_table_name <- "catalog_40_copper_statistics_services.analytics_app.ees_publications"

sc <- spark_connect(method = "databricks")

In [0]:
%r
scraped_publications <- sparklyr::sdf_sql(sc, paste("SELECT * FROM", scrape_table_name)) |> collect()

In [0]:
%r
pub_context <- tibble::tribble(
    ~title, ~division, ~latest_release_type, ~frequency, ~newsworthiness,
    
    # --- Alphabetised publications ---
    "16-18 destination measures", "DISD", "OS", "Annual + update with revised data", "Low",
    "A level and other 16 to 18 results", "DISD", "AOS", "Annual + update with revised data", "High",
    "Academy transfers and funding", "Academies", "OS", "Annual", "Unknown",
    "Admission appeals in England", "DISD", "AOS", "Annual", "Low",
    "Apprenticeships", "DISD", "AOS", "Monthly", "Medium",
    "Apprenticeships and 19-plus Further Education Skills Index", "Skills Policy", "OS", "Annual", "Low",
    "Apprenticeships and traineeships", "DISD", "AOS ", "Superseded", "-","-",
    "Apprenticeships in England by industry characteristics", "Skills Policy", "OSiD", "Annual", "TBA",
    "Career pathways: post-16 qualifications held by employees", "Ad hoc", "TBA", "TBA", "TBA",
    "Childcare and early years provider survey", "FAD", "OS", "Annual", "TBA",
    "Childcare and early years survey of parents", "FAD", "OS", "Annual", "Low",
    "Children accommodated in secure children's homes", "DISD", "AOS", "Annual", "Medium",
    "Children in need", "DISD", "AOS", "Annual", "Medium",
    "Children in need: A focus on re-referrals", "SAPAD", "Ad hoc", "-", "Unknown",
    "Children in need: A focus on sexual abuse and exploitation", "SAPAD", "Ad hoc", "-", "Unknown",
    "Children looked after in England including adoptions", "DISD", "AOS", "Annual + update to add stability data", "Medium",
    "Children looked after: A focus on placement location", "TBA", "TBA", "TBA", "TBA",
    "Children missing education", "DISD", "OSiD", "Annual", "Medium",
    "Children's social work workforce", "DISD", "OS", "Annual", "Medium",
    "Children's social work workforce: attrition, caseload, and agency workforce", "Ad hoc", "TBA", "TBA", "TBA",
    "CO2 monitors: cumulative delivery statistics", "MI", "TBA", "TBA", "TBA",
    "Coronavirus (COVID-19) Reporting in Higher Education Providers", "Ad hoc", "TBA", "TBA", "TBA",
    "COVID mass testing data in education", "Ad hoc", "TBA", "TBA", "TBA",
    "Delivery of air cleaning units", "MI", "TBA", "TBA", "TBA",
    "Detailed destinations of 16 to 18 year olds in Further Education", "Ad hoc", "TBA", "TBA", "TBA",
    "Early years education recovery", "TBA", "MI", "TBA", "TBA",
    "Early years foundation stage profile results", "DISD", "AOS", "Annual", "Medium",
    "ECF and NPQ starts", "MI", "TBA", "TBA", "TBA",
    "Education and training statistics for the UK", "International", "AOS", "Annual", "Medium",
    "Education, children’s social care and offending: local authority level dashboard", "SAPAD", "Ad hoc", "TBA", "TBA",
    "Education, health and care plans", "DISD", "AOS", "Annual", "High",
    "Education provision: children under 5 years of age", "DISD", "TBA", "TBA", "TBA",
    "Elective home education", "DISD", "OSiD", "Annual", "Medium",
    "Employer Skills Survey", "Skills Policy", "OS", "Once every 2 years", "Medium",
    "Estimate of additional children claiming Free School Meals following expansion of eligibility", "RFDA", "TBA", "TBA", "TBA",
    "Expansion to early childcare entitlements: Childcare Experiences Survey", "SAPAD", "TBA", "TBA", "TBA",
    "Expansion to early childcare entitlements: eligibility codes issued and validated", "SAPAD", "MI", "Termly", "Unknown",
    "FE learners going into employment and learning destinations by local authority district", "TBA", "TBA", "TBA", "TBA",
    "Foundation year participation, provision and outcomes at HE providers", "Ad hoc", "TBA", "TBA", "TBA",
    "Free school meals: Autumn term", "TBA", "TBA", "TBA", "TBA",
    "Funded early education and childcare", "DISD", "Ad hoc", "One off", "TBA",
    "Further education and skills", "DISD", "AOS", "Quarterly", "Medium",
    "Further education skills index", "TBA", "TBA", "TBA", "TBA",
    "Further education: outcome-based success measures", "TBA", "TBA", "TBA", "TBA",
    "Further education outcomes", "Skills Policy", "OS", "Annual", "Low",
    "Further education workforce", "DISD", "OSiD", "Annual", "High",
    "Graduate labour market statistics", "HE", "OS", "Annual + update to add stability data", "Low",
    "Graduate outcomes (LEO)", "TBA", "TBA", "TBA", "TBA",
    "Graduate outcomes (LEO): postgraduate outcomes", "TBA", "TBA", "TBA", "TBA",
    "Higher Education Entrants and Qualifiers by their Level 2 and 3 Attainment", "Ad hoc", "TBA", "TBA", "TBA",
    "Higher Level Learners in England", "TBA", "TBA", "TBA", "TBA",
    "Home to school transport: LA data collection", "TBA", "TBA", "TBA", "TBA",
    "Initial Teacher Training Census", "TAD", "OS", "Annual", "TBA",
    "Initial teacher training performance profiles", "TAD", "OS", "Annual", "Medium",
    "Key stage 1 and phonics screening check attainment", "TBA", "TBA", "TBA", "TBA",
    "Key stage 2 attainment", "DISD", "AOS", "Annual + update with revised data", "Medium",
    "Key stage 2 attainment: National headlines", "DISD", "AOS", "Annual", "High",
    "Key stage 4 destination measures", "DISD", "OS", "Annual + update with revised data", "Low",
    "Key stage 4 performance", "DISD", "AOS", "Annual + update with revised data", "High",
    "LA and school expenditure", "DISD", "OS", "Annual", "High",
    "Laptops and tablets data", "OS", "stopped already", "TBA", "TBA",
    "LEO Graduate and Postgraduate Outcomes", "HE", "OS", "Annual", "TBA",
    "LEO Graduate outcomes provider level data", "HE", "OS", "Annual", "Medium",
    "Level 2 and 3 attainment age 16 to 25", "DISD", "AOS", "Annual", "Medium",
    "Local authority school places scorecards", "DISD", "OS", "Annual", "Low",
    "Longer term destinations", "DISD", "OS", "Annual", "Low",
    "Looked after children aged 16 to 17 in independent or semi-independent placements", "DISD", "Ad hoc", "One off", "TBA",
    "Multi-academy trust performance measures (Key stages 2, 4 and 5)", "TBA", "TBA", "TBA", "TBA",
    "Multi-academy trust performance measures at key stage 2", "TBA", "TBA", "TBA", "TBA",
    "Multiplication tables check attainment", "DISD", "OS", "Annual", "Medium",
    "National pupil projections", "DISD", "OS", "Annual", "Medium",
    "National Tutoring Programme", "TAD", "OS", "Various", "Medium",
    "NEET age 16 to 24", "DISD", "OSID", "Annual", "Medium",
    "Occupations in demand", "TBA", "TBA", "TBA", "TBA",
    "Outcomes for children in need, including children looked after by local authorities in England", "DISD", "AOS", "Annual", "High",
    "Outcomes of children in need, including looked after children", "TBA", "TBA", "TBA", "TBA",
    "Parental responsibility measures", "DISD", "OS", "Annual", "Medium",
    "Participation in education, training and employment age 16 to 18", "DISD", "AOS", "Annual", "Medium",
    "Participation in education, training and NEET age 16 to 17 by local authority", "DISD", "MI", "Annual", "TBA",
    "Participation measures in higher education", "HE", "OS", "Annual", "High",
    "Permanent exclusions and suspensions in England", "TBA", "TBA", "TBA", "TBA",
    "Phonics screening check attainment", "TBA", "TBA", "TBA", "TBA",
    "Planned LA and school expenditure", "DISD", "OS", "Annual", "High",
    "Postgraduate initial teacher training targets", "TAD", "OS", "Annual", "Medium",
    "Primary and secondary school applications and offers", "TBA", "TBA", "TBA", "TBA",
    "Progression to higher education or training", "DISD", "OS", "Annual", "TBA",
    "Provisional T Level results", "DISD", "OS", "One off", "High",
    "Pupil absence in schools in England", "DISD", "AOS", "Termly", "Medium",
    "Pupil absence in schools in England: autumn and spring terms", "TBA", "TBA", "TBA", "TBA",
    "Pupil absence in schools in England: autumn term", "TBA", "TBA", "TBA", "TBA",
    "Pupil attendance in schools", "DISD", "OSiD", "Fortnightly", "High",
    "Pupil yield from housing developments", "DISD", "MI", "Annual", "TBA",
    "School capacity", "DISD", "OS", "Annual", "High",
    "School counts by average Progress 8 scores for disadvantaged White British pupils", "TBA", "TBA", "TBA", "TBA",
    "School finances during the Covid-19 pandemic", "Ad hoc", "TBA", "TBA", "TBA",
    "School funding statistics", "SAFA", "OS", "Annual + update to add stability data", "Unknown",
    "School Leadership retention", "TBA", "TBA", "TBA", "TBA",
    "School placements for children from outside of the UK", "TBA", "TBA", "TBA", "TBA",
    "School places sufficiency survey", "DISD", "Ad hoc", "One off", "TBA",
    "School workforce in England", "DISD", "AOS", "Annual", "High",
    "Schools eligible for RISE intervention", "TBA", "TBA", "TBA", "TBA",
    "Schools, pupils and their characteristics", "DISD", "AOS", "Annual", "Medium",
    "Secondary and primary school applications and offers", "TBA", "TBA", "TBA", "TBA",
    "September Guarantee: offers of education and training for young people age 16 and 17", "Not DISD", "MI", "TBA", "TBA",
    "Serious incident notifications", "DISD", "OS", "Annual", "High",
    "Skills Bootcamps outcomes", "TBA", "TBA", "TBA", "TBA",
    "Skills Bootcamps starts", "TBA", "TBA", "TBA", "TBA",
    "Skills bootcamps starts, completions and outcomes", "TBA", "TBA", "TBA", "TBA",
    "Special educational needs in England", "DISD", "AOS", "Annual", "Medium",
    "Stability measures for children looked after in England", "TBA", "TBA", "TBA", "TBA",
    "Student loan forecasts for England", "HE", "OS", "Annual", "TBA",
    "Supply of skills for jobs in science and technology", "Ad hoc", "TBA", "TBA", "TBA",
    "Suspensions and permanent exclusions in England", "DISD", "AOS", "Termly", "High",
    "Teacher and leader development: ECF and NPQs", "TAD", "OSiD", "Annual", "Medium",
    "The link between absence and attainment at KS2 and KS4", "DISD", "Ad hoc", "One off", "TBA",
    "UK revenue from education related exports and transnational education activity", "HE", "OSiD", "Annual", "Medium",
    "Vulnerable children and young people survey", "MI", "TBA", "TBA", "TBA",
    "Widening participation in higher education", "HE", "OS", "Annual", "Medium",
    "Attendance in education and early years settings during the coronavirus (COVID-19) pandemic", "TBA", "TBA", "TBA", "TBA"
  )

pub_context <- pub_context %>% arrange(title)

In [0]:
%r
publications_with_context <- scraped_publications %>%
  dplyr::left_join(pub_context, by = "title") 

There are some duplicate titles in the scrape data (for good reasons) so I will deduplicate those for the test

In [0]:
%r
test_that("Checking total publication number has not changed", {
  missing_in_context <- setdiff(scraped_publications$title, pub_context$title)
  missing_in_scrape <- setdiff(pub_context$title, scraped_publications$title)
  info_msg <- paste0(
    "Publication row count mismatch: You need to manually update the pub_context table to add or remove new publication details.\n",
    "Publications missing in context: ", paste(sort(missing_in_context), collapse = ", "), "\n",
    "Publications missing in scrape: ", paste(sort(missing_in_scrape), collapse = ", ")
  )
  expect_equal(n_distinct(scraped_publications$title), nrow(pub_context),
    info = info_msg
  )
})

In [0]:
%r
test_that("Warn if any TBA values exist in publications_with_context", {
  tba_issues <- publications_with_context %>%
    filter(if_any(everything(), ~ . == "TBA"))
  if (nrow(tba_issues) > 0) {
    warning_msg <- paste0(
      "TBA values found in publications_with_context for the following titles:\n",
      paste(tba_issues$title, collapse = ", ")
    )
    warning(warning_msg)
    # Print titles for rows with TBA
    cat("Titles with TBA values:\n", paste(tba_issues$title, collapse = ", "), "\n")
    # Print slugs for titles with TBA
    tba_slugs <- tba_issues %>%
      mutate(slug = stringr::str_replace_all(tolower(title), "[^a-z0-9]+", "-")) %>%
      pull(slug)
    cat("Slugs for titles with TBA:\n", paste(tba_slugs, collapse = ", "), "\n")
  }
  expect_true(nrow(tba_issues) == 0, info = "There are still TBA values in publications_with_context.")
})

In [0]:
%r
updated_spark_df <- copy_to(sc, publications_with_context, overwrite = TRUE)

# Write to temp table while we confirm we're good to overwrite data
spark_write_table(updated_spark_df, paste0(write_table_name, "_temp"), mode = "overwrite")

temp_table_data <- sparklyr::sdf_sql(sc, paste0("SELECT * FROM ", write_table_name, "_temp")) %>% collect()
previous_data <- tryCatch(
  {
    sparklyr::sdf_sql(sc, paste0("SELECT * FROM ", write_table_name)) %>% collect()
  },
  error = function(e) {
    NULL
  }
)

test_that("Temp table data matches updated data", {
  expect_equal(nrow(temp_table_data), nrow(publications_with_context))
})

# Replace the old table with the new one
dbExecute(sc, paste0("DROP TABLE IF EXISTS ", write_table_name))
dbExecute(sc, paste0("ALTER TABLE ", write_table_name, "_temp RENAME TO ", write_table_name))

print_changes_summary(temp_table_data, previous_data)